# Module 3 — Wind & Ocean Current Pipeline (SLICKTRACE)**Owner:** NajmaThis notebook fetches wind (ERA5/CDS) and ocean current (CMEMS) data for a spill location/time and outputs `environment.json` for Module 4 (OpenDrift).**Before running:** register at https://data.marine.copernicus.eu/register (CMEMS) and https://cds.climate.copernicus.eu (CDS), and accept the ERA5 dataset license once on the CDS website.Run cells top to bottom. Cell 6 is a **fallback mock generator** — use it if either API is slow/down so you're never blocked.

In [ ]:
# CELL 1 — Install dependencies
!pip install -q copernicusmarine cdsapi xarray netCDF4 numpy pandas

In [ ]:
# CELL 2 — Demo point + config# Pulls spill location from Module 1, falls back to demo values.## Priority order:#   1. spill_input.json from GitHub (pushed by Module 1 or manually)#   2. Module 1 metadata files in metadata/ directory (S1_*.json)#   3. Hardcoded demo valuesGITHUB_RAW_URL = "https://raw.githubusercontent.com/ShuraShipai/maris/main/integration/spill_input.json"GITHUB_METADATA_URL = "https://api.github.com/repos/ShuraShipai/maris/contents/module1_data/metadata"FALLBACK_LAT = -18.00FALLBACK_LON = 147.00FALLBACK_TIME = "2022-07-15T06:00:00"BUFFER_DEG = 0.5OUTPUT_PATH = "environment.json"import json, requests, glob, osfrom datetime import datetime, timedeltaimport numpy as np, pandas as pd, xarray as xrdef load_spill_input():    """Try GitHub spill_input.json, then Module 1 metadata, then demo fallback."""    # --- Attempt 1: spill_input.json from GitHub ---    try:        resp = requests.get(GITHUB_RAW_URL + "?t=" + str(datetime.utcnow().timestamp()), timeout=10)        resp.raise_for_status()        data = resp.json()        lat, lon = float(data["latitude"]), float(data["longitude"])        time_str = data["timestamp"]        print("[Source: spill_input.json] lat=" + str(lat) + ", lon=" + str(lon) + ", time=" + time_str)        if "image_id" in data:            print("  (image: " + data["image_id"] + ")")        return lat, lon, time_str    except Exception as e:        print("[WARN] spill_input.json not available (" + str(e) + ")")    # --- Attempt 2: Module 1 metadata files from GitHub ---    try:        resp = requests.get(GITHUB_METADATA_URL + "?t=" + str(datetime.utcnow().timestamp()), timeout=10)        resp.raise_for_status()        files = resp.json()        # Find the latest S1_*.json file        s1_files = [f for f in files if f["name"].startswith("S1_") and f["name"].endswith(".json")]        if s1_files:            latest = sorted(s1_files, key=lambda x: x["name"])[-1]            meta_resp = requests.get(latest["download_url"], timeout=10)            meta_resp.raise_for_status()            meta = meta_resp.json()            lat = float(meta["latitude"])            lon = float(meta["longitude"])            date_str = meta["date"]  # e.g. "2026-08-15"            time_str = date_str + "T06:00:00"            print("[Source: Module 1 metadata (" + latest["name"] + ")] lat=" + str(lat) + ", lon=" + str(lon) + ", time=" + time_str)            if "image_id" in meta:                print("  (image: " + meta["image_id"] + ")")            return lat, lon, time_str    except Exception as e:        print("[WARN] Module 1 metadata not available (" + str(e) + ")")    # --- Attempt 3: Demo fallback ---    print("Using hardcoded demo values.")    return FALLBACK_LAT, FALLBACK_LON, FALLBACK_TIMEDEMO_LAT, DEMO_LON, DEMO_TIME = load_spill_input()print("Using: lat=" + str(DEMO_LAT) + ", lon=" + str(DEMO_LON) + ", time=" + DEMO_TIME)

## CredentialsUse Colab Secrets (key icon in left sidebar) to store `CMEMS_USER`, `CMEMS_PASS`, `CDS_KEY` — do NOT hardcode them in a cell you might share.If Secrets aren't set up yet, `getpass` prompts will fall back so you're never blocked.

In [ ]:
# CELL 3 — Load credentials (Colab Secrets, with getpass fallback)
import getpass

try:
    from google.colab import userdata

    CMEMS_USER = userdata.get("CMEMS_USER")
    CMEMS_PASS = userdata.get("CMEMS_PASS")
    CDS_KEY = userdata.get("CDS_KEY")

except Exception:
    CMEMS_USER = None
    CMEMS_PASS = None
    CDS_KEY = None

if not CMEMS_USER:
    CMEMS_USER = input("CMEMS username/email: ")

if not CMEMS_PASS:
    CMEMS_PASS = getpass.getpass("CMEMS password: ")

if not CDS_KEY:
    CDS_KEY = getpass.getpass(
        "CDS API key (from cds.climate.copernicus.eu/api): "
    )

print("Credentials loaded.")


In [ ]:
# CELL 4 — Ocean currents (CMEMS)
# CELL 4 — Ocean currents (CMEMS)
import copernicusmarine

def get_currents(lat, lon, time_str, buffer=BUFFER_DEG):
    """Return (current_u, current_v) in m/s at the given point/time."""

    t = pd.Timestamp(time_str)

    ds = copernicusmarine.open_dataset(
        dataset_id="cmems_mod_glo_phy_anfc_0.083deg_PT1H-m",
        minimum_longitude=lon - buffer,
        maximum_longitude=lon + buffer,
        minimum_latitude=lat - buffer,
        maximum_latitude=lat + buffer,
        start_datetime=(t - timedelta(hours=3)).isoformat(),
        end_datetime=(t + timedelta(hours=3)).isoformat(),
        username=CMEMS_USER,
        password=CMEMS_PASS,
    )

    point = ds.interp(
        latitude=lat,
        longitude=lon,
        time=t,
        method="linear",
    )

    # Surface layer = first depth index if a depth dimension exists
    if "depth" in point.dims:
        point = point.isel(depth=0)

    u = float(point["uo"].values)
    v = float(point["vo"].values)

    return u, v


In [ ]:
# CELL 5 — Wind (ERA5 via CDS)
# CELL 5 — Wind (ERA5 via CDS)
import cdsapi

cds_client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    key=CDS_KEY,
)

def get_wind(lat, lon, time_str, buffer=BUFFER_DEG):
    """Return (wind_u, wind_v) in m/s at 10m height for the given point/time."""

    t = pd.Timestamp(time_str)
    target = "era5_wind_tmp.nc"

    cds_client.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
            ],
            "year": f"{t.year}",
            "month": f"{t.month:02d}",
            "day": f"{t.day:02d}",
            "time": f"{t.hour:02d}:00",
            "area": [
                lat + buffer,
                lon - buffer,
                lat - buffer,
                lon + buffer,
            ],  # North, West, South, East
            "data_format": "netcdf",
        },
        target,
    )

    ds = xr.open_dataset(target)

    point = ds.interp(
        latitude=lat,
        longitude=lon,
        method="linear",
    )

    u = float(point["u10"].values)
    v = float(point["v10"].values)

    return u, v


In [ ]:
# CELL 6 — FALLBACK: mock generator (use if APIs are slow/down)

import random

def get_mock_environment(lat, lon, time_str):
    """Physically plausible placeholder values so Member 4 is never blocked."""

    return {
        "time": time_str,
        "latitude": lat,
        "longitude": lon,
        "wind_u": round(random.uniform(-8, 8), 2),
        "wind_v": round(random.uniform(-8, 8), 2),
        "current_u": round(random.uniform(-0.5, 0.5), 2),
        "current_v": round(random.uniform(-0.5, 0.5), 2),
    }


# Uncomment to test the fallback immediately:
# print(json.dumps(
#     get_mock_environment(DEMO_LAT, DEMO_LON, DEMO_TIME),
#     indent=2
# ))


In [ ]:
# CELL 7 — Build environment.json (real data, with automatic fallback)

def build_environment_json(lat, lon, time_str, use_fallback_on_error=True):
    try:
        current_u, current_v = get_currents(
            lat,
            lon,
            time_str,
        )

        wind_u, wind_v = get_wind(
            lat,
            lon,
            time_str,
        )

        result = {
            "time": time_str,
            "latitude": lat,
            "longitude": lon,
            "wind_u": wind_u,
            "wind_v": wind_v,
            "current_u": current_u,
            "current_v": current_v,
        }

    except Exception as e:
        print(
            f"[WARN] Live fetch failed ({e}).",
            "Using mock fallback." if use_fallback_on_error else "",
        )

        if not use_fallback_on_error:
            raise

        result = get_mock_environment(
            lat,
            lon,
            time_str,
        )

    with open(OUTPUT_PATH, "w") as f:
        json.dump(result, f, indent=2)

    print(f"Wrote {OUTPUT_PATH}:")
    print(json.dumps(result, indent=2))

    return result


# Run it for the agreed demo point
env = build_environment_json(
    DEMO_LAT,
    DEMO_LON,
    DEMO_TIME,
)


## Sanity checkWind should be roughly **0–20 m/s**, currents roughly **0–2 m/s**. If you see values wildly outside this, it's almost always a units or interpolation bug (e.g. picked the wrong depth layer, or grid mismatch), not a real ocean condition.

In [ ]:
# CELL 8 — Sanity check

def sanity_check(env):
    checks = []

    checks.append(("wind_u", -30, 30))
    checks.append(("wind_v", -30, 30))
    checks.append(("current_u", -3, 3))
    checks.append(("current_v", -3, 3))

    ok = True

    for key, lo, hi in checks:
        val = env[key]

        status = "OK" if lo <= val <= hi else "SUSPICIOUS"

        if status == "SUSPICIOUS":
            ok = False

        print(f"{key}: {val}  [{status}]")

    print(
        "\nAll values look physically reasonable."
        if ok
        else "\nCheck flagged values above before handing off."
    )


sanity_check(env)


In [ ]:
# CELL 9 — Save to Google Drive (auto-save every run)from google.colab import drivedrive.mount("/content/drive")!cp environment.json /content/drive/MyDrive/environment.jsonprint("Saved to Drive: /content/drive/MyDrive/environment.json")

## Handoff to Module 4`environment.json` is now in the notebook's working directory (and optionally Drive). Send this file to Member 4, or commit it to `module3_environment/` in the repo:```{  "time": "...",  "latitude": ...,  "longitude": ...,  "wind_u": ...,  "wind_v": ...,  "current_u": ...,  "current_v": ...}```Confirm with Member 4 that sign convention (positive = eastward/northward) and units (m/s) match what OpenDrift expects.

In [ ]:
# CELL 10 — Auto-commit environment.json to GitHub## ONE-TIME SETUP:#   1. Go to https://github.com/settings/tokens#   2. Generate new token (classic), check "repo" scope#   3. Replace YOUR_TOKEN below with the token you copied#   4. Replace YOUR_EMAIL with your GitHub emailGITHUB_TOKEN = "YOUR_TOKEN"  # <-- PASTE YOUR TOKEN HEREGITHUB_EMAIL = "YOUR_EMAIL"  # <-- YOUR GITHUB EMAIL!git config --global user.email "" + GITHUB_EMAIL + ""!git config --global user.name "Najma"# Clone and setup!git clone https://github.com/ShuraShipai/maris.git 2>/dev/null || true%cd maris!git checkout module3 2>/dev/null || git checkout -b module3# Copy the latest environment.json!cp /content/environment.json integration/environment.json# Commit and push!git add integration/environment.json!git commit -m "auto: update environment.json from Module 3 pipeline" 2>/dev/null || echo "No changes to commit"!git remote set-url origin https://" + GITHUB_TOKEN + "@github.com/ShuraShipai/maris.git!git push origin module3%cd ..print("Done! environment.json committed to module3 branch")

## Handoff to Module 4 is now:1. In your Colab working directory2. Saved to Google Drive (Cell 9)3. Committed to the  branch on GitHub (Cell 10)Module 4 can pull the latest output from:Confirm with Module 4: units = m/s, positive = eastward/northward.